# Network as diluted faculae (sweep_network.dag)

In [ ]:
import glob, json, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sunsim import observations as obs

BASE_NW = '/data/spotless/scratch/sstucki/starsim/network'
TAG = 'network_m45s_size_4096_N60'
SETS = os.path.join(BASE_NW, 'sets_' + TAG)
IDENT = os.path.join(SETS, 'ident')
START = '2015-07-18'                             # t = 0 of the simulations
OBS = '/data/spotless/scratch/sstucki/starsim/observations'
HARPSN = os.path.join(OBS, 'harpn_sun_release_timeseries_2015-2025.csv')
INTERVENTIONS = os.path.join(OBS, 'harpsn_intervention_days.txt')
SORCE, TSIS = os.path.join(OBS, 'DataSSI - SORCE.sav'), os.path.join(OBS, 'tsis_tsi_24hr.csv')

PARTS = ('spot', 'plage', 'network', 'network_fraction', 'spot_plage', 'spot_plage_network',
         'spot_plage_network_fraction')
TOTALS = ('spot_plage', 'spot_plage_network', 'spot_plage_network_fraction')
NETWORKS = ('network', 'network_fraction')
COLOURS = {'spot': 'k', 'plage': 'C3', 'network': 'C1', 'network_fraction': 'C4', 'spot_plage': 'C0',
           'spot_plage_network': 'C2', 'spot_plage_network_fraction': 'C9'}
MAX_FF_FC, MAX_FF_SP = 20.0, 5.0                 # epochs left out (bad HMI images), as obs.sweep_stats
NOISE_MODEL = 'corr'                             # 'corr' (lag-1 correlation of the day) or 'white'
MAX_CONTAMINATION = 0.05                         # noise pixels / network pixels accepted
TAU = {'SORCE/TIM': 27.0, 'TSIS-1': 27.0, 'HARPS-N': 1.0}   # correlation time of the residuals [d] (errors)
HIGHPASS = 81.0                                  # running mean removed before the high-pass fits [d]
NAME_RE = re.compile(r'noise_th_(?P<noise_th>[\d.]+)_B_th_(?P<B_th>[\d.]+)_I_th_(?P<I_th>[\d.]+)_A_th_(?P<A_th>[\d.]+)'
                     r'_min_px_(?P<min_px>\d+)_B_sat_(?P<B_sat>[\d.]+)_T_(?P<T_sp>\d+)_')


def params_of(name):
    return {k: float(v) for k, v in NAME_RE.match(name).groupdict().items()}


def label_of(p):
    return 'noise%g_B%g_I%g_A%g_min%g_Bsat%g' % (p['noise_th'], p['B_th'], p['I_th'], p['A_th'], p['min_px'],
                                                 p['B_sat'])


def short(name):
    p = params_of(name)
    return 'B_th %g, I_th %.2f, min_px %d' % (p['B_th'], p['I_th'], p['min_px'])


NAMES = sorted(os.path.basename(f) for f in glob.glob(os.path.join(SETS, 'spot_plage', '*')) if os.path.isdir(f))
DONE = [n for n in NAMES if all(os.path.exists(os.path.join(SETS, p, n, 'rv.txt')) for p in PARTS)]
SETS_DF = pd.DataFrame([dict(params_of(n), name=n) for n in DONE])
DIAG = sorted(os.path.basename(f) for f in glob.glob(os.path.join(IDENT, 'diag', '*')) if
              os.path.exists(os.path.join(f, 'histograms.npz')))
print(len(DONE), 'of', len(NAMES), 'sets simulated;', 'noise diagnostics:', *DIAG)
SETS_DF

## Noise of the magnetograms

In [ ]:
H = {k: dict(np.load(os.path.join(IDENT, 'diag', k, 'histograms.npz'))) for k in DIAG}
h = H[DIAG[0]]
dates = pd.to_datetime(h['date'].astype(str))
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={'width_ratios': [1, 2]})
mu = h['mu_centres'][0]
q16, q50, q84 = np.nanpercentile(h['sigma'], [16, 50, 84], axis=0)
ax.fill_between(mu, q16, q84, color='C0', alpha=0.3)
ax.plot(mu, q50, 'o-', c='C0')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$\sigma$ [G] (1.4826 MAD)')
ax2.plot(dates, h['rho1'], '.', ms=3, c='C0', label=r'$\rho_1$ of B / $\sigma$ (|B| < 3$\sigma$)')
ax2.plot(dates, h['s_kernel'], '.', ms=3, c='C1', label='kernel of the noise [px]')
ax2.set_ylabel(r'$\rho_1$, kernel [px]')
ax2.legend(markerscale=4)
fig.tight_layout()
plt.show()

## Bright groups of every size: data and pure noise

In [ ]:
def area_px():
    """Largest group of the network [px]: the median area_th_pixels of the sets (A_th projected at mu = 0.1)."""
    f = sorted(glob.glob(os.path.join(IDENT, 'noise*_min*', 'summary.csv')))
    return int(np.floor(np.median(pd.read_csv(f[0])['area_th_pixels']))) if f else 22


A_PX = area_px()
sizes = np.arange(1, int(H[DIAG[0]]['S_MAX']) + 1)
fig, axes = plt.subplots(1, len(DIAG), figsize=(4.2 * len(DIAG), 3.8), sharey=True, squeeze=False)
for ax, k in zip(axes[0], DIAG):
    h = H[k]
    nd = h['n_disc'][:, None]
    for key, c, lab in (('n_pix', 'k', 'data'), ('white_pix', 'C0', 'white noise'), ('corr_pix', 'C3', 'correlated noise')):
        ax.step(sizes, 100 * np.mean(h[key] / nd, axis=0), where='mid', c=c, label=lab)
    ax.axvline(A_PX + 0.5, c='0.5', ls='--', lw=1)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('group size [px]')
    ax.set_title('B_th = %g G' % h['B_th'])
axes[0, 0].set_ylabel('pixels of the groups [% of the disc]')
axes[0, 0].legend()
fig.tight_layout()
plt.show()

## Noise contamination of the network against min_px

In [ ]:
M = np.arange(1, A_PX + 1)


def contamination(h, model=NOISE_MODEL):
    """Noise pixels / data pixels of the groups of m ... A_PX pixels, every day (rows) and m (columns)."""
    cn = np.cumsum(h[model + '_pix'][:, :A_PX][:, ::-1], axis=1)[:, ::-1]
    cd = np.cumsum(h['n_pix'][:, :A_PX][:, ::-1], axis=1)[:, ::-1]
    return cn / np.maximum(cd, 1)


activity = {k: H[k]['n_pix'][:, -1] / H[k]['n_disc'] for k in DIAG}   # groups >= S_MAX px: plage
rows = []
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
for i, k in enumerate(DIAG):
    h = H[k]
    for ax, model in zip(axes, ('white', 'corr')):
        c = contamination(h, model)
        lo, hi = activity[k] <= np.median(activity[k]), activity[k] > np.median(activity[k])
        ax.plot(M, np.median(c[lo], axis=0), '-', c='C%d' % i, label='B_th %g G, low activity' % h['B_th'])
        ax.plot(M, np.median(c[hi], axis=0), '--', c='C%d' % i, label='B_th %g G, high activity' % h['B_th'])
    c = contamination(h)
    q = np.quantile(c, 0.84, axis=0)                     # most of the days below MAX_CONTAMINATION
    m_best = int(M[np.argmax(q < MAX_CONTAMINATION)]) if np.any(q < MAX_CONTAMINATION) else np.nan
    pix = h['n_pix'][:, :A_PX] / h['n_disc'][:, None]
    rows.append({'diag': k, 'B_th': float(h['B_th']), 'min_px': m_best,
                 'contamination_min_px_0': np.median(c[:, 0]),
                 'network_ff_min_px_0': 100 * np.median(pix.sum(axis=1)),
                 'network_ff_min_px': 100 * np.median(pix[:, int(m_best) - 1:].sum(axis=1)) if m_best == m_best else np.nan})
for ax, model in zip(axes, ('white', 'corr')):
    ax.axhline(MAX_CONTAMINATION, c='0.5', ls=':')
    ax.set_yscale('log')
    ax.set_xlabel('min_px')
    ax.set_title('%s noise' % model)
axes[0].set_ylabel('noise / network pixels')
axes[1].legend(fontsize=8, ncol=2)
fig.tight_layout()
plt.show()
BEST = pd.DataFrame(rows)
BEST

## Network filling factor and dilution

In [ ]:
def ident_summary(p):
    s = pd.read_csv(os.path.join(IDENT, label_of(p), 'summary.csv'))
    s['date'] = pd.to_datetime(s['date'].astype(str).str[:10])
    return s


fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
for i, (_, r) in enumerate(SETS_DF[np.isclose(SETS_DF['I_th'], 0.9)].iterrows()):
    s = ident_summary(r)
    lab = 'B_th %g, min_px %d' % (r['B_th'], r['min_px'])
    axes[0].plot(s['date'], s['ff_net'], '.', ms=2, c='C%d' % i, label=lab)
    axes[0].plot(s['date'], s['ff_net_frac'], 'x', ms=2, c='C%d' % i)
    axes[1].plot(s['date'], s['ff_net_frac'] / s['ff_net'], '.', ms=2, c='C%d' % i, label=lab)
s = ident_summary(SETS_DF.iloc[0])
axes[0].plot(s['date'], s['ff_plage'], '-', lw=1, c='C3', alpha=0.5, label='plage')
axes[0].set_ylabel('ff [% of the disc] (dots: full, x: fraction)')
axes[1].set_ylabel('ff fraction / ff full')
axes[0].legend(markerscale=5, ncol=4, fontsize=8)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
for i, k in enumerate(DIAG):
    h = H[k]
    with np.errstate(invalid='ignore'):
        f = np.sum(h['frac'], axis=0) / np.sum(h['n_pix'], axis=0)
    ax.step(sizes, f, where='mid', c='C%d' % i, label='B_th %g G' % h['B_th'])
ax.axvline(A_PX + 0.5, c='0.5', ls='--', lw=1)
ax.set_xscale('log')
ax.set_xlabel('group size [px] (last bin: >= %d)' % sizes[-1])
ax.set_ylabel('mean min(|B|/mu/B_sat, 1) per pixel')
ax.set_title('B_sat = %g G' % 200)
ax.legend()
plt.show()

## Light curve and RV of the parts

In [ ]:
def series(part, name):
    """Time series of one part (obs._read_series) with its disc filling factors (summary.csv), the bad epochs of
    spot_plage_network left out."""
    s = obs._read_series(os.path.join(SETS, part, name))
    sm = pd.read_csv(os.path.join(SETS, part, name, 'summary.csv'))
    i = np.searchsorted(sm['t'].values, s['time'])
    s['disc_sp'], s['disc_fc'] = sm['ff_sp'].values[i], sm['ff_fc'].values[i]
    ref = obs._read_series(os.path.join(SETS, 'spot_plage_network', name))
    bad = set(ref['time'][(ref['ff_fc'] > MAX_FF_FC) | (ref['ff_sp'] > MAX_FF_SP)])
    keep = ~np.isin(s['time'], list(bad))
    return {k: v[keep] for k, v in s.items()}


def ppm(lc):
    return 1e6 * (lc / np.nanmean(lc) - 1)


ok = BEST.set_index('B_th')['min_px'].to_dict()
d = SETS_DF[np.isclose(SETS_DF['I_th'], 0.9)].copy()
d['ok'] = [r['min_px'] >= ok.get(r['B_th'], np.inf) for _, r in d.iterrows()]
CHOSEN = d.sort_values(['ok', 'B_th', 'min_px'], ascending=[False, True, True]).iloc[0]['name']
print('chosen:', short(CHOSEN))
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
for p in PARTS:
    s = series(p, CHOSEN)
    axes[0].plot(s['time'], ppm(s['lc']), '.', ms=1.5, c=COLOURS[p], label=p)
    axes[1].plot(s['time'], s['rv'] - np.nanmedian(s['rv']), '.', ms=1.5, c=COLOURS[p], label=p)
axes[0].set_ylabel('Flux [ppm]')
axes[1].set_ylabel('RV [m/s]')
axes[1].set_xlabel('Time [days since %s]' % START)
axes[0].legend(markerscale=6, ncol=4, fontsize=8)
fig.suptitle(short(CHOSEN))
fig.tight_layout()
plt.show()

## Contribution of every part

In [ ]:
def efficiency(x, y):
    """Slope of y against x (least squares with a constant)."""
    ok = np.isfinite(x) & np.isfinite(y)
    return np.polyfit(x[ok], y[ok], 1)[0] if ok.sum() > 2 else np.nan


rows = []
for name in DONE:
    for p in PARTS:
        s = series(p, name)
        ff = s['disc_fc'] if p in ('plage', 'network', 'network_fraction') else s['disc_sp'] + s['disc_fc']
        rows.append(dict(params_of(name), part=p, lc_std=np.nanstd(ppm(s['lc'])), rv_std=np.nanstd(s['rv']),
                         ff_mean=np.nanmean(ff), lc_per_ff=efficiency(ff, ppm(s['lc'])),
                         rv_per_ff=efficiency(ff, s['rv'])))
CONTRIB = pd.DataFrame(rows)
for value, label in (('lc_std', 'std of the flux [ppm]'), ('rv_std', 'std of the RV [m/s]'),
                     ('lc_per_ff', 'flux per % of the disc [ppm]'), ('rv_per_ff', 'RV per % of the disc [m/s]')):
    t = CONTRIB.pivot_table(index=['B_th', 'I_th', 'min_px'], columns='part', values=value)
    t = t[[p for p in PARTS if p in t]] if value.endswith('std') else t[['plage', 'network', 'network_fraction']]
    ax = t.plot.bar(figsize=(12, 3.5), rot=0, color=[COLOURS[c] for c in t.columns], width=0.85)
    ax.set_ylabel(label)
    ax.legend(fontsize=8, ncol=4)
    plt.show()

## Totals against HARPS-N and TSI

In [ ]:
rows = []
for part in TOTALS:
    for name in DONE:
        f = os.path.join(SETS, part, name, 'stats.json')
        if os.path.exists(f):
            rows.append(dict(json.load(open(f)), part=part, name=name))
STATS = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (value, label) in zip(axes, (('rv_rms', 'RV rms vs HARPS-N [m/s]'), ('sorce_rms', 'TSI rms vs SORCE [ppm]'),
                                     ('tsis_rms', 'TSI rms vs TSIS-1 [ppm]'))):
    for part in TOTALS:
        for B_th, mk in zip(sorted(STATS['B_th'].unique()), 'osD^'):
            g = STATS[(STATS['part'] == part) & (STATS['B_th'] == B_th) & np.isclose(STATS['I_th'], 0.9)]
            g = g.sort_values('min_px')
            ax.plot(g['min_px'], g[value], marker=mk, c=COLOURS[part], label='%s, B_th %g' % (part, B_th))
    ax.set_xlabel('min_px')
    ax.set_ylabel(label)
axes[0].legend(fontsize=7)
fig.tight_layout()
plt.show()
STATS[['part', 'B_th', 'I_th', 'min_px', 'rv_rms', 'sorce_rms', 'tsis_rms', 'fwhm_rms', 'bis_rms']].sort_values(
    ['B_th', 'I_th', 'min_px', 'part']).round(3)

## Network amplitude fitted on the observations

In [ ]:
harps = obs.load_harpsn(HARPSN, START)
tsis_all = {'SORCE/TIM': obs.load_sorce(SORCE, START), 'TSIS-1': obs.load_tsis(TSIS, START)}
cuts = obs.load_interventions(INTERVENTIONS, START)


def running_mean(t, y, width):
    """Mean of y over the points within width / 2 of every t (t sorted)."""
    c = np.concatenate(([0.0], np.cumsum(y)))
    lo, hi = np.searchsorted(t, t - width / 2), np.searchsorted(t, t + width / 2, side='right')
    return (c[hi] - c[lo]) / (hi - lo)


def fit(y, X, groups, tau, free_plage=True):
    """y = offsets of the groups + a X[:, 0] + alpha X[:, 1] (a = 1 if not free_plage). Returns a, alpha, their errors
    (residual variance, times tau for correlated residuals), rms of the residuals with alpha and with alpha = 0."""
    G = (groups[:, None] == np.unique(groups)[None, :]).astype(float)
    target = y if free_plage else y - X[:, 0]
    cols = X if free_plage else X[:, 1:]
    out = []
    for A in (np.column_stack([cols, G]), np.column_stack([cols[:, :-1], G])):
        beta, *_ = np.linalg.lstsq(A, target, rcond=None)
        res = target - A @ beta
        out.append((beta, res, A))
    (beta, res, A), (_, res0, _) = out
    cov = np.linalg.pinv(A.T @ A) * np.var(res) * tau
    err = np.sqrt(np.diag(cov))
    a, a_err = (beta[0], err[0]) if free_plage else (1.0, 0.0)
    k = 1 if free_plage else 0
    return a, a_err, beta[k], err[k], obs.rms(res), obs.rms(res0)


def design(name, net, quantity):
    """(y, X, groups, t) of every observation of quantity ('HARPS-N' or a key of tsis_all) matched by day, X: the
    simulated spot_plage and the network part net."""
    sp, nw = series('spot_plage', name), series(net, name)
    assert np.array_equal(sp['time'], nw['time'])
    if quantity == 'HARPS-N':
        io, i_s = obs.match_days(harps['t'], sp['time'])
        y = harps['rv'][io]
        X = np.column_stack([sp['rv'][i_s], nw['rv'][i_s]])
        groups = obs._periods(harps['t'][io], cuts)
    else:
        tsi = tsis_all[quantity]
        io, i_s = obs.match_days(tsi['t'], sp['time'])
        y = ppm(tsi['tsi'][io])
        X = np.column_stack([ppm(sp['lc'])[i_s], ppm(nw['lc'])[i_s]])
        groups = np.zeros(len(io), int)
    ok = np.isfinite(y) & np.all(np.isfinite(X), axis=1)
    return y[ok], X[ok], groups[ok], sp['time'][i_s][ok]


rows = []
for name in DONE:
    p = params_of(name)
    dil = ident_summary(p)
    dilution = np.sum(dil['ff_net_frac']) / np.sum(dil['ff_net'])
    for net in NETWORKS:
        for quantity in ('SORCE/TIM', 'TSIS-1', 'HARPS-N'):
            y, X, groups, t = design(name, net, quantity)
            for filt in ('full', 'high-pass'):
                yy, XX = y, X
                if filt == 'high-pass':
                    yy = y - running_mean(t, y, HIGHPASS)
                    XX = X - np.column_stack([running_mean(t, X[:, j], HIGHPASS) for j in range(X.shape[1])])
                for free in (True, False):
                    a, a_err, alpha, alpha_err, rms1, rms0 = fit(yy, XX, groups, TAU[quantity], free)
                    rows.append(dict(p, name=name, network=net, quantity=quantity, filter=filt, plage_free=free,
                                     a=a, a_err=a_err, alpha=alpha, alpha_err=alpha_err, rms=rms1, rms_no_network=rms0,
                                     n=len(yy), dilution=dilution))
FITS = pd.DataFrame(rows)
FITS[(FITS['filter'] == 'full') & FITS['plage_free']][
    ['network', 'quantity', 'B_th', 'I_th', 'min_px', 'a', 'a_err', 'alpha', 'alpha_err', 'rms', 'rms_no_network',
     'dilution']].sort_values(['network', 'B_th', 'I_th', 'min_px', 'quantity']).round(3)

In [ ]:
MARKERS = {'SORCE/TIM': 'o', 'TSIS-1': 's', 'HARPS-N': 'D'}
QCOL = {'SORCE/TIM': 'C0', 'TSIS-1': 'C2', 'HARPS-N': 'C3'}
for filt in ('full', 'high-pass'):
    fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharex=True)
    for c, net in enumerate(NETWORKS):
        for r, free in enumerate((True, False)):
            ax = axes[r, c]
            for B_th, ls in zip(sorted(FITS['B_th'].unique()), ('-', '--', ':', '-.')):
                for quantity in MARKERS:
                    g = FITS[(FITS['network'] == net) & (FITS['quantity'] == quantity) & (FITS['filter'] == filt)
                             & (FITS['plage_free'] == free) & (FITS['B_th'] == B_th) & np.isclose(FITS['I_th'], 0.9)]
                    g = g.sort_values('min_px')
                    ax.errorbar(g['min_px'] + 0.1 * list(MARKERS).index(quantity), g['alpha'], g['alpha_err'],
                                marker=MARKERS[quantity], ls=ls, c=QCOL[quantity], capsize=2,
                                label='%s, B_th %g' % (quantity, B_th))
                if net == 'network':
                    g = FITS[(FITS['B_th'] == B_th) & np.isclose(FITS['I_th'], 0.9)].drop_duplicates('min_px')
                    g = g.sort_values('min_px')
                    ax.plot(g['min_px'], g['dilution'], ls=ls, c='k', label='ff fraction / ff full, B_th %g' % B_th)
            ax.axhline(1, c='0.5', lw=1)
            ax.axhline(0, c='0.5', lw=1, ls=':')
            ax.set_title('%s, plage %s' % (net, 'free' if free else 'fixed (a = 1)'), loc='left', fontsize=10)
            ax.set_ylabel(r'$\alpha$ of the network')
    for ax in axes[-1]:
        ax.set_xlabel('min_px')
    axes[0, 0].legend(fontsize=7, ncol=2)
    fig.suptitle('%s (I_th = 0.9)' % filt)
    fig.tight_layout()
    plt.show()

### Fits of the chosen set

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
for ax, quantity in zip(axes, ('SORCE/TIM', 'TSIS-1', 'HARPS-N')):
    y, X, groups, t = design(CHOSEN, 'network', quantity)
    G = (groups[:, None] == np.unique(groups)[None, :]).astype(float)
    off = G @ np.linalg.lstsq(np.column_stack([X, G]), y, rcond=None)[0][2:]
    ax.plot(t, y - off, '.', ms=2, c='0.6', label=quantity)
    for net in NETWORKS:
        f = FITS[(FITS['name'] == CHOSEN) & (FITS['network'] == net) & (FITS['quantity'] == quantity)
                 & (FITS['filter'] == 'full') & FITS['plage_free']].iloc[0]
        _, Xn, _, _ = design(CHOSEN, net, quantity)
        ax.plot(t, f['alpha'] * Xn[:, 1], '.', ms=2, c=COLOURS[net],
                label=r'%s $\times$ %.2f' % (net, f['alpha']))
    ax.plot(t, X[:, 0], '.', ms=1, c=COLOURS['spot_plage'], alpha=0.5, label='spot_plage')
    ax.set_ylabel('[ppm]' if quantity != 'HARPS-N' else '[m/s]')
    ax.legend(markerscale=5, ncol=4, fontsize=8)
axes[-1].set_xlabel('Time [days since %s]' % START)
fig.suptitle(short(CHOSEN))
fig.tight_layout()
plt.show()